# Week 8 Coding Practice Solution: Working with Time Series in pandas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/DataScience-book/blob/w08-cp-01-colab-link/module08/week8_time_series_practice_solution.ipynb)

This solution notebook shows one clean workflow for investigating the bike-count time series. It is meant to guide interpretation and checking, not to replace your own reasoning.

## Learning objectives

- parse timestamps and inspect a DatetimeIndex;
- select date ranges;
- compare resampled frequencies;
- interpret rolling and lagged relationships; and
- evaluate a simple baseline forecast chronologically.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')

candidate_paths = [
    Path('appliedDataScience/module08/fremont_bridge_daily.csv'),
    Path('fremont_bridge_daily.csv'),
    Path.cwd() / 'appliedDataScience/module08/fremont_bridge_daily.csv',
]
data_path = next((p for p in candidate_paths if p.exists()), candidate_paths[0])
if not data_path.exists():
    raise FileNotFoundError(f'Missing dataset: {data_path}')

bridge = pd.read_csv(data_path, parse_dates=['date']).sort_values('date').set_index('date')
bridge = bridge.rename(columns={'bikes': 'daily_count'})
bridge_daily = bridge.asfreq('D')
print(bridge_daily.head())
print(f'Rows: {len(bridge_daily)}')
print(f'Start: {bridge_daily.index.min()}')
print(f'End: {bridge_daily.index.max()}')
print(f'Missing dates: {bridge_daily["daily_count"].isna().sum()}')


In [ ]:
subset = bridge_daily.loc['2023-01-01':'2023-01-31']
print(subset[['daily_count']].head())
print(f'Mean January 2023 count: {subset["daily_count"].mean():.1f}')


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
bridge_daily['daily_count'].plot(ax=ax, color='#2c7fb8', linewidth=1.2)
ax.set(title='Daily bicycle counts', xlabel='Date', ylabel='Count')
plt.tight_layout()
plt.show()


In [ ]:
rolling_7 = bridge_daily['daily_count'].rolling(window=7, min_periods=1).mean()
fig, ax = plt.subplots(figsize=(12, 4))
bridge_daily['daily_count'].plot(ax=ax, alpha=0.7, label='Daily count')
rolling_7.plot(ax=ax, label='7-day rolling mean', color='#d95f02', linewidth=2.5)
ax.legend()
ax.set(title='Seven-day summary', xlabel='Date', ylabel='Count')
plt.tight_layout()
plt.show()


In [ ]:
weekly_total = bridge_daily['daily_count'].resample('W-MON').sum()
monthly_average = bridge_daily['daily_count'].resample('MS').mean()
print(weekly_total.head())
print(monthly_average.head())


In [ ]:
weekday_means = bridge_daily.groupby(bridge_daily.index.day_name())['daily_count'].mean()
ordered_days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_means = weekday_means.reindex(ordered_days)
print(weekday_means)


In [ ]:
bridge_daily['lag_1'] = bridge_daily['daily_count'].shift(1)
bridge_daily['lag_7'] = bridge_daily['daily_count'].shift(7)
print(bridge_daily[['daily_count', 'lag_1', 'lag_7']].head(10))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(bridge_daily['lag_1'], bridge_daily['daily_count'], alpha=0.25, color='#7570b3')
axes[0].set(title='Lag 1', xlabel='Previous day count', ylabel='Current day count')
axes[1].scatter(bridge_daily['lag_7'], bridge_daily['daily_count'], alpha=0.25, color='#e7298a')
axes[1].set(title='Lag 7', xlabel='Previous week count', ylabel='Current day count')
for ax in axes:
    ax.axline((0, 0), slope=1, color='black', linewidth=1, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
train = bridge_daily.iloc[:-28].copy()
test = bridge_daily.iloc[-28:].copy()
naive = pd.Series(train['daily_count'].iloc[-1], index=test.index, name='naive')
seasonal_pattern = train['daily_count'].tail(7).to_numpy()
seasonal_naive = pd.Series(
    np.tile(seasonal_pattern, int(np.ceil(len(test) / 7)) + 1)[: len(test)],
    index=test.index,
    name='seasonal_naive',
)
forecast_df = pd.concat([test['daily_count'], naive, seasonal_naive], axis=1)
forecast_df.columns = ['actual', 'naive', 'seasonal_naive']
print(forecast_df.head())
mae_naive = mean_absolute_error(forecast_df['actual'], forecast_df['naive'])
rmse_naive = np.sqrt(mean_squared_error(forecast_df['actual'], forecast_df['naive']))
mae_seasonal = mean_absolute_error(forecast_df['actual'], forecast_df['seasonal_naive'])
rmse_seasonal = np.sqrt(mean_squared_error(forecast_df['actual'], forecast_df['seasonal_naive']))
print(f'Naive MAE: {mae_naive:.1f}')
print(f'Naive RMSE: {rmse_naive:.1f}')
print(f'Seasonal naive MAE: {mae_seasonal:.1f}')
print(f'Seasonal naive RMSE: {rmse_seasonal:.1f}')


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
forecast_df['actual'].plot(ax=ax, label='Actual', color='#1b9e77', linewidth=1.5)
forecast_df['naive'].plot(ax=ax, label='Naive', linestyle='--', color='#d95f02')
forecast_df['seasonal_naive'].plot(ax=ax, label='Seasonal naive', linestyle=':', color='#7570b3')
ax.set(title='Chronological forecast comparison', xlabel='Date', ylabel='Daily count')
ax.legend()
plt.tight_layout()
plt.show()


## Interpretation notes

- The series has a clear daily index, so `asfreq('D')` and date filtering are meaningful.
- Rolling windows smooth the series and help reveal short-term trend without removing all variation.
- A 7-day lag captures weekly repeating structure better than a 1-day lag when counts are strongly tied to the weekly calendar.
- A seasonal-naive forecast often performs better than a flat naive forecast when day-of-week structure is present.
- A chronological split is essential: we evaluate the forecast using only information available at the cut-off time.